# LUMENOIR — Kaggle evaluation

This notebook evaluates the existing LUMENOIR pipeline on the permanent Kaggle dataset `/kaggle/input/datasets/chiranth25x/lumenoir-low-light-datasets`, the ExDark mount, the DICM mount, and datasets transferred into `/kaggle/working`.

**Before running**
1. Attach `lumenoir-low-light-datasets` at `/kaggle/input/datasets/chiranth25x/lumenoir-low-light-datasets`.
2. ExDark is at `/kaggle/input/datasets/washingtongold/exdark-dataset`; DICM is at `/kaggle/input/datasets/twozeb/dicm-img`.
3. SICE and NPE/NPE-ex1/NPE-ex2/NPE-ex3, LIME, MEF and VV are not assumed to be permanent Kaggle datasets. Upload/extract them into `/kaggle/working` (or an extracted temporary root) when needed.
4. Attach the LUMENOIR repository and the Zero-DCE++ checkpoint (`zero_dce_plus_plus.pth` or `Epoch99.pth`).
5. Enable a GPU accelerator if you want faster Zero-DCE++ inference; classical stages stay on CPU.

LOL-v1, LOL-v2 Real and LOL-v2 Synthetic are discovered with both train and test pairs; only test/eval records are evaluated. SICE uses the Hugging Face `train/samples/*/{Input,GT}.jpg` and `test/samples/*/{Input,GT}.jpg` layout; only its test samples are evaluated. NPE/NPE-ex1/NPE-ex2/NPE-ex3 are grouped as one logical `npe` dataset (85 images), with subgroup metadata. LIME (10 BMP), MEF (17 PNG), NPE (85) and VV (24) are unpaired, so PSNR/SSIM/LPIPS are never computed for them. JPG, JPEG, PNG and BMP are supported.

The runner never downloads datasets, tunes parameters or fine-tunes the model.

In [ ]:
import os
import sys
from pathlib import Path

# Attach the repository as a Kaggle dataset or clone it here.
REPO = Path("/kaggle/working/LUMENOIR")
if not REPO.is_dir():
    REPO = Path("/kaggle/input/lumenoir-code/LUMENOIR")
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("Repository:", REPO)

In [ ]:
# NumPy/OpenCV/Pillow are required. LPIPS/NIQE need the optional metrics stack.
# !pip install -q -r requirements-eval.txt
# !pip install -q -r requirements-metrics.txt
print("Install the requirement files above if the imports below fail.")

## 1. Inspect dataset discovery

This writes deterministic manifests under `/kaggle/working/lumenoir-evaluation/manifests`.
It does not load the model or compute metrics.

In [ ]:
from training.kaggle_eval.cli import main

main([
    "--discover-only",
    "--input-root", "/kaggle/input/datasets/chiranth25x/lumenoir-low-light-datasets",
    "--additional-root", "/kaggle/input/datasets/washingtongold/exdark-dataset",
    "--additional-root", "/kaggle/input/datasets/twozeb/dicm-img",
    "--additional-root", "/kaggle/working",
    "--additional-root", "/kaggle/working/lumenoir_additional/extracted",
    "--output-dir", "/kaggle/working/lumenoir-evaluation",
])

## 2. Configure and run

Copy `training/kaggle_eval/example_config.json` to `/kaggle/working/eval_config.json` and edit:

- `additional_roots` for the permanent low-light mount, ExDark, DICM and `/kaggle/working` (where SICE/NPE/LIME/MEF/VV may be transferred);
- dataset `root`, `pairs` and `images` paths; NPE auto-groups from `/kaggle/working`;
- the checkpoint `weights` path;
- `device` (`auto`, `cpu`, `cuda`, `mps`);
- `skip_metrics` if LPIPS/NIQE dependencies are unavailable.

Explicit pair directories are required when auto-discovery is ambiguous. Missing references are skipped and logged; paired metrics are never computed without a ground-truth image. Training records are discovered for counts but are never evaluated.

In [ ]:
import json
from pathlib import Path

from training.kaggle_eval.config import EvalConfig
from training.kaggle_eval.runner import run_evaluation

CONFIG_PATH = Path("/kaggle/working/eval_config.json")
if not CONFIG_PATH.is_file():
    CONFIG_PATH.write_text(
        Path("training/kaggle_eval/example_config.json").read_text(),
        encoding="utf-8",
    )

config = EvalConfig.from_file(CONFIG_PATH)
summary = run_evaluation(config)
print(json.dumps(summary.to_dict(), indent=2))

## 3. Inspect the reports

The aggregate Markdown contains dataset/mode/metric summaries. Per-image JSONL records include metric notes and per-stage timings.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

output = Path(config.output_dir)
display(Markdown((output / "aggregate.md").read_text(encoding="utf-8")))

In [ ]:
from pathlib import Path

from IPython.display import Image, display

visuals = sorted((Path(config.output_dir) / "visuals").rglob("*.png"))
print(f"{len(visuals)} before/after images")
for path in visuals[:3]:
    display(Image(filename=str(path)))

## Limitations

- Unpaired datasets (ExDark, DICM, NPE including NPE-ex1/2/3, LIME, MEF, VV) have no ground truth; only no-reference and input-referenced artifact metrics are computed. PSNR/SSIM/LPIPS are never computed for them.
- NPE and its three extensions appear as one `npe` dataset (85 images); `records.jsonl`/`records.csv` retain the `subgroup` and the manifest reports per-subgroup counts.
- SICE is read from `train/samples/*/{Input,GT}.jpg` and `test/samples/*/{Input,GT}.jpg`; only test samples are evaluated, never training samples.
- LOL/SICE training records are discovered for counts but filtered out of the evaluation set.
- SICE Part 1 is Zero-DCE++ pre-training data; paired SICE results are not held out.
- Paired images must have identical dimensions; mismatches are logged as failures.
- LPIPS and NIQE depend on optional packages and their model files; skip them on offline runs.
- No quality improvement should be claimed until these reports are populated with real datasets.